# Bayesian Hyperparameter Space Optimization and Automated Model Auditing Pipeline

In [1]:
%load_ext autoreload
%autoreload 2

import os
import mlflow
import optuna
from optuna.integration.mlflow import MLflowCallback
from optuna.pruners import MedianPruner
import numpy as np
import pandas as pd

from src import feature_engineering as fe
from src import optuna_optimization as utils
from src import mlflow_utils as mlf_utils

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Path Configuration & MLflow Backend Binding

In [2]:
RANDOM_STATE = 42
N_SPLITS = 5
EXPERIMENT_NAME = "customer-churn-optuna"

from pathlib import Path

def _find_project_root():
    """Find the project root by looking for pyproject.toml."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find project root (pyproject.toml)")

ROOT_DIR = str(_find_project_root())
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

# Set the tracking URI immediately to lock it to SQLite
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# Explicitly create the experiment with your designated mlartifacts folder path
experiment_id = mlf_utils.init_mlflow_experiment(
    EXPERIMENT_NAME,
    DB_PATH,
    ARTIFACTS_DIR,
)

# Active the experiment scope
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='file:///Users/hector.vargas/repos/ml_hands_on_project/mlartifacts', creation_time=1781781840960, experiment_id='1', last_update_time=1781781840960, lifecycle_stage='active', name='customer-churn-optuna', tags={}, trace_location=None, workspace='default'>

## 2. Custom Source Code Ingestion

In [3]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

## 2. Orchestrate and Initialize Search Parameter Studies

In [4]:
selected_features = [
    # Binary
    "is_silver",
    "is_germany",
    "is_spain",
    "Num_Of_Products_1",
    "Num_Of_Products_2",
    "Num_Of_Products_3",
    "Num_Of_Products_4",

    # Continuous
    "Age_x_IsActive",
    "Balance_per_Product",
    "CreditScore_per_Age",
    "Inactive_x_Balance",
    "CreditScore_x_Age",
    "Products_per_Tenure",
]
# Schema Baseline Columns Definitions
nomod_columns = []
dummyfy_columns = ['Gender']
norm_std_columns = ['Point Earned', 'Satisfaction Score', 'EstimatedSalary']

# Initialize the Feature Engineer class with the desired subset strings
feature_engineer_object = fe.DynamicFeatureEngineer(selected_features=selected_features)
binary_features = feature_engineer_object._get_all_binary_features()
continuous_features = feature_engineer_object._get_all_continuous_features()

current_layout = {
    "passthrough": nomod_columns + binary_features,
    "standard_scale": norm_std_columns + continuous_features,
    "one_hot_encode": dummyfy_columns
}

EXPERIMENT_REGISTRY = {
    "experiment_1": current_layout
}

# Initialize Integrated MLflow Tracking Integration Callbacks
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="pr_auc",
    create_experiment=False,
    mlflow_kwargs={
        "nested": True,
        "experiment_id": experiment_id,
    },
)

study = optuna.create_study(
    study_name="customer-churn-rf-search-v1",
    direction="maximize",
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=0),
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

# FIXED: Removed the invalid FULL_REGISTRY argument variable completely
# Instantiate the objective function with both training and testing datasets
objective_function = utils.ObjectiveCV(
    X=X_train,
    y=y_train,
    current_layout=current_layout,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
)

/var/folders/bv/50x24wc545x5mclk_t88ryrc0000gn/T/ipykernel_19579/580893370.py:40: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
[I 2026-06-18 05:51:05,184] A new study created in memory with name: customer-churn-rf-search-v1


## 3. Run Optimization Search Workspace Execution Window

In [ ]:
# Execute the parent run context block
with mlflow.start_run(
    run_name="optuna_search_parent",
    experiment_id=experiment_id,
):    
    study.optimize(
        objective_function,
        n_trials=5000,
        callbacks=[mlflow_callback]
    )

    # Log global baseline metrics directly to the parent run metadata shell
    mlflow.log_metric("best_auc", study.best_value)
    mlflow.log_params(study.best_params)

    # Reconstruct the pipeline architecture using the best parameters found
    best_pipeline = utils.build_pipeline(
        trial=study.best_trial, 
        current_layout=current_layout,
        random_state=RANDOM_STATE
    )

    # Run full evaluations and save serialization schema artifacts to the parent run
    utils.evaluate_and_log_best_model(
        best_pipeline=best_pipeline,
        X_train=X_train, 
        y_train=y_train,
        X_test=X_test, 
        y_test=y_test,
        current_layout=current_layout,
    )

[I 2026-06-18 05:51:08,381] Trial 0 finished with value: 0.6788379596098221 and parameters: {'scaler': 'minmax', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 293, 'rf_max_depth': 12, 'rf_min_samples_split': 25, 'rf_min_samples_leaf': 1}. Best is trial 0 with value: 0.6788379596098221.
[I 2026-06-18 05:51:09,934] Trial 1 finished with value: 0.6953347996684984 and parameters: {'scaler': 'std', 'encoder': 'no_drop', 'model': 'xgb', 'xgb_n_estimators': 1059, 'xgb_learning_rate': 0.029667626364545993, 'xgb_subsample': 0.9644741157888952, 'xgb_colsample_bytree': 0.47092407909780626, 'xgb_gamma': 2.0842892970704363, 'xgb_reg_alpha': 6.78905327169848e-07, 'xgb_reg_lambda': 1.9069966103000426e-07, 'xgb_scale_pos_weight': 1.992587980696507}. Best is trial 1 with value: 0.6953347996684984.
[I 2026-06-18 05:51:12,115] Trial 2 finished with value: 0.6800919420716183 and parameters: {'scaler': 'robust', 'encoder': 'no_drop', 'model': 'rf', 'rf_n_estimators': 295, 'rf_max_depth': 13, '

## 4. Display Session Optimization Diagnostics Results

In [9]:
print(f"\nTop Optimization Average Precision Score Target: {study.best_value:.4f}")
print("\nOptimal Parameter Combinations Selected:")
for parameter_key, parameter_value in study.best_params.items():
    print(f" * {parameter_key}: {parameter_value}")


Top Optimization Average Precision Score Target: 0.7011

Optimal Parameter Combinations Selected:
 * scaler: robust
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 1193
 * xgb_learning_rate: 0.02666397513885387
 * xgb_subsample: 0.9405664371553261
 * xgb_colsample_bytree: 0.5071531693840389
 * xgb_gamma: 3.3272862165562556
 * xgb_reg_alpha: 3.5662055338761855e-07
 * xgb_reg_lambda: 1.57208112167439e-08
 * xgb_scale_pos_weight: 1.674451432575116


In [10]:
suggestions = utils.suggest_numeric_ranges(study)
display(suggestions)

,parameter,current_min,current_max,best_median,action,suggested_min,suggested_max
0,xgb_learning_rate,2.500081e-02,0.038387,2.641775e-02,move_left,0.018308,0.038387
1,xgb_reg_alpha,1.034129e-08,0.000976,3.356325e-07,move_left,-0.000488,0.000976
2,xgb_reg_lambda,1.002093e-09,0.000097,1.869154e-08,move_left,-0.000048,0.000097
3,xgb_scale_pos_weight,1.600003e+00,2.093801,1.629352e+00,move_left,1.353103,2.093801
4,xgb_subsample,9.400002e-01,0.979183,9.408030e-01,move_left,0.920409,0.979183
5,xgb_gamma,1.946135e+00,3.499981,3.409834e+00,move_right,1.946135,4.276904
6,xgb_colsample_bytree,4.500792e-01,0.595474,5.143019e-01,narrow,0.499548,0.522617
7,xgb_n_estimators,8.000000e+02,1400.000000,9.840000e+02,narrow,913.240000,1051.560000


In [12]:
runs_df = mlflow.search_runs()
# Select rows where 'tags.mlflow.parentRunId' is missing (meaning it IS a parent/master run)
parent_summary_df = runs_df[runs_df["tags.mlflow.parentRunId"].isna() & (runs_df["status"] == "FINISHED")]

# Isolate evaluation metrics
metric_cols = [c for c in parent_summary_df.columns if c.startswith("metrics.")]
display(parent_summary_df[["start_time"] + metric_cols])

,start_time,metrics.pr_auc,metrics.test_recall,metrics.train_f1,metrics.train_pr_auc,metrics.test_accuracy,metrics.train_roc_auc,metrics.best_auc,metrics.test_pr_auc,metrics.test_f1,metrics.train_recall,metrics.test_roc_auc,metrics.train_precision,metrics.test_precision,metrics.train_accuracy
5000,2026-06-18 11:51:05.297000+00:00,NaN,0.598039,0.719371,0.833148,0.8650,0.940743,0.701076,0.736358,0.643799,0.666411,0.876276,0.781475,0.697143,0.894062
5011,2026-06-18 11:28:51.412000+00:00,NaN,0.583333,0.740313,0.849821,0.8625,0.948050,0.696080,0.733325,0.633822,0.688650,0.874386,0.800357,0.693878,0.901563
5022,2026-06-18 11:24:36.134000+00:00,NaN,0.583333,0.740313,0.849821,0.8625,0.948050,0.696080,0.733325,0.633822,0.688650,0.874386,0.800357,0.693878,0.901563


In [13]:
run_id = parent_summary_df.iloc[0]["run_id"]

run = mlflow.get_run(run_id)

print(run.data.metrics)

{'best_auc': 0.7010758060322027, 'train_accuracy': 0.8940625, 'train_precision': 0.7814748201438849, 'train_recall': 0.6664110429447853, 'train_f1': 0.7193708609271523, 'train_roc_auc': 0.9407429500823452, 'train_pr_auc': 0.8331477466491667, 'test_accuracy': 0.865, 'test_precision': 0.6971428571428572, 'test_recall': 0.5980392156862745, 'test_f1': 0.6437994722955145, 'test_roc_auc': 0.8762762956941571, 'test_pr_auc': 0.736357614669004}


In [14]:
parent_summary_df

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.pr_auc,metrics.test_recall,metrics.train_f1,metrics.train_pr_auc,...,tags.xgb_colsample_bytree_distribution,tags.model_distribution,tags.xgb_learning_rate_distribution,tags.xgb_subsample_distribution,tags.encoder_distribution,tags.xgb_gamma_distribution,tags.rf_max_depth_distribution,tags.rf_n_estimators_distribution,tags.rf_min_samples_leaf_distribution,tags.rf_min_samples_split_distribution
5000,525c024b77e645028f4616c00f34b044,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 11:51:05.297000+00:00,2026-06-18 12:57:00.865000+00:00,NaN,0.598039,0.719371,0.833148,...,None,None,None,None,None,None,None,None,None,None
5011,5f2fd1dc86194a0f89a9555b199fa9cf,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 11:28:51.412000+00:00,2026-06-18 11:29:09.127000+00:00,NaN,0.583333,0.740313,0.849821,...,None,None,None,None,None,None,None,None,None,None
5022,912ffa2b5d69402e85318243adf7ce03,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 11:24:36.134000+00:00,2026-06-18 11:24:53.337000+00:00,NaN,0.583333,0.740313,0.849821,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
# from src import feature_engineering as fe
# fe.remove_recent_runs(EXPERIMENT_NAME, 100000)

In [17]:
rf_runs = (
    parent_summary_df[parent_summary_df["params.model"] == "rf"]
    .sort_values(
        "metrics.train_pr_auc",
        ascending=False
    )
)

rf_runs[
    [
        "run_id",
        'metrics.train_pr_auc',
        "metrics.pr_auc",
        "params.rf_max_depth",
        "params.rf_n_estimators",
        "params.rf_min_samples_split",
        "params.rf_min_samples_leaf",
    ]
]

,run_id,metrics.train_pr_auc,metrics.pr_auc,params.rf_max_depth,params.rf_n_estimators,params.rf_min_samples_split,params.rf_min_samples_leaf


In [ ]:
xgb_runs = (
    parent_summary_df[parent_summary_df["params.model"] == "xgb"]
    .sort_values(
        "metrics.train_pr_auc",
        ascending=False
    )
)

xgb_runs[
    [
        "run_id",
        "metrics.train_pr_auc",
        "metrics.test_pr_auc",
        "metrics.pr_auc",
        "params.xgb_n_estimators",
        "params.xgb_learning_rate",
        "params.xgb_gamma",
        "params.xgb_scale_pos_weight",
    ]
].head(20)

,run_id,metrics.train_pr_auc,metrics.test_pr_auc,metrics.pr_auc,params.xgb_n_estimators,params.xgb_learning_rate,params.xgb_gamma,params.xgb_scale_pos_weight
5011,5f2fd1dc86194a0f89a9555b199fa9cf,0.849821,0.733325,NaN,1264,0.02809743391955222,2.9137146876952342,1.6370223258670453
5022,912ffa2b5d69402e85318243adf7ce03,0.849821,0.733325,NaN,1264,0.02809743391955222,2.9137146876952342,1.6370223258670453
5000,525c024b77e645028f4616c00f34b044,0.833148,0.736358,NaN,1193,0.02666397513885387,3.3272862165562556,1.674451432575116
